# 2.2 多算子调用 vs FlashAttention

## 小节概述

运行 Notebook 前设置以下环境变量，不在代码中填写 `/path/to/...` 占位符：

<table style="text-align:left; margin-left:0;">
<tr><th>变量</th><th>要求</th></tr>
<tr><td><code>MUDUOXINYU_ROOT</code></td><td>一次性工作副本；HEAD 为课程精确基线，工作树干净</td></tr>
<tr><td><code>MUDUOXINYU_MODEL</code></td><td>合法取得的模型文件</td></tr>
<tr><td><code>MUDUOXINYU_TOKENIZER</code></td><td>与模型配套的 tokenizer 文件</td></tr>
<tr><td><code>MUDUOXINYU_OUTPUT</code></td><td>本次专用且尚不存在的输出目录</td></tr>
</table>

模型和 tokenizer 不随课程仓库分发。两者必须来自同一套实验资产；本节记录路径、大小与 SHA256，但不会下载或覆盖资产。


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess


def require_path(env_name, kind):
    value = os.environ.get(env_name, "").strip()
    if not value:
        raise RuntimeError(f"请先设置环境变量 {env_name}")
    path = Path(value).expanduser().resolve()
    valid = path.is_dir() if kind == "dir" else path.is_file()
    if not valid:
        raise FileNotFoundError(f"{env_name} 指向的{kind}不存在：{path}")
    return path


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


MUDUO_ROOT = require_path("MUDUOXINYU_ROOT", "dir")
MODEL = require_path("MUDUOXINYU_MODEL", "file")
TOKENIZER = require_path("MUDUOXINYU_TOKENIZER", "file")
output_value = os.environ.get("MUDUOXINYU_OUTPUT", "").strip()
if not output_value:
    raise RuntimeError("请先设置环境变量 MUDUOXINYU_OUTPUT")
OUTPUT = Path(output_value).expanduser().resolve()
if OUTPUT.exists():
    raise FileExistsError(f"输出目录必须尚不存在：{OUTPUT}")
SRC = Path("src").resolve()
BASELINE_COMMIT = "60c6371cd30894d9896dfa979b86c6f892b6cbda"

print("MuduoXinyu:", MUDUO_ROOT)
print("Model:", MODEL, "bytes=", MODEL.stat().st_size, "sha256=", sha256(MODEL))
print("Tokenizer:", TOKENIZER, "bytes=", TOKENIZER.stat().st_size, "sha256=", sha256(TOKENIZER))
print("Output:", OUTPUT)


## 1. 确认设备、精确基线与一次性工作副本

先查看 NPU，再验证 MuduoXinyu HEAD、工作树和旧构建产物。补丁脚本还会重复检查，不会为通过检查而执行 reset、clean 或 checkout。


In [ ]:
subprocess.run(["npu-smi", "info"], check=True)
commit = subprocess.check_output(
    ["git", "-C", str(MUDUO_ROOT), "rev-parse", "HEAD"], text=True
).strip()
status = subprocess.check_output(
    ["git", "-C", str(MUDUO_ROOT), "status", "--porcelain", "--untracked-files=normal"],
    text=True,
).strip()
if commit != BASELINE_COMMIT:
    raise RuntimeError(f"HEAD={commit}，预期基线={BASELINE_COMMIT}")
if status:
    raise RuntimeError(f"工作树不干净，课程不会丢弃这些改动：\n{status}")
if (MUDUO_ROOT / "muduoXinyu").exists():
    raise RuntimeError("检测到旧 muduoXinyu 构建产物；请改用新的、一次性工作副本")
print("BASELINE_STATUS=PASS commit=", commit)


## 2. 补丁 dry-run 与应用

第一次调用只执行哈希、工作树和 `git apply --check`，不写入目标仓库；第二次才应用补丁并校验关键文件哈希。重复执行第二次会明确报告补丁已应用，不会 reset 或覆盖。


In [ ]:
subprocess.run([
    "bash", str(SRC / "apply_patch.sh"),
    "--muduo-root", str(MUDUO_ROOT), "--check-only",
], check=True)
subprocess.run([
    "bash", str(SRC / "apply_patch.sh"),
    "--muduo-root", str(MUDUO_ROOT),
], check=True)


## 3. 构建 NPU 版本

工作副本在补丁前已经确认没有旧二进制，因此直接构建即可。课程不执行 `make clean`；若构建失败，保留当前工作树和日志供排查，不自动删除或回退。


In [ ]:
ascend = os.environ.get("ASCEND_HOME") or os.environ.get("ASCEND_HOME_PATH") or os.environ.get("ASCEND_PATH")
if not ascend:
    raise RuntimeError("请设置 ASCEND_HOME 为 CANN Toolkit 根目录")
subprocess.run(["make", "-C", str(MUDUO_ROOT), "npu", f"ASCEND_PATH={ascend}"], check=True)
binary = MUDUO_ROOT / "muduoXinyu"
if not binary.is_file():
    raise FileNotFoundError(f"构建结束但未找到二进制：{binary}")
print("BUILD_STATUS=PASS binary=", binary)


## 4. 实现包 A/B 运行

两条路径固定相同模型、prompt、steps、temperature、topP 和 Device，但实现包本身包含两项差异：

```bash
# Path A：FP32 多算子 Attention
./muduoXinyu MODEL TOKENIZER INPUT --backend npu --enableDeviceOpt \
  --skipValidation --temperature 0 --topP 1 --dumpTokenSummary --steps 120

# Path B：FP16 aclnnIncreFlashAttentionV4 + 必要 Cast
./muduoXinyu MODEL TOKENIZER INPUT --backend npu --enableDeviceOpt \
  --skipValidation --temperature 0 --topP 1 --dumpTokenSummary --steps 120 \
  --useNpuFlashAttention
```

runner 先按 A→B 运行带 profiling 的 smoke，再反转为 B→A 运行正式性能，保存完整命令、退出码、Device、commit、工作树 diff SHA256、模型/tokenizer SHA256 和四份原始日志。顺序反转只能减小固定顺序偏差，不能把复合实现比较变成纯单变量实验。


In [ ]:
subprocess.run([
    "bash", str(SRC / "run_ab_benchmark.sh"),
    "--muduo-root", str(MUDUO_ROOT),
    "--model", str(MODEL),
    "--tokenizer", str(TOKENIZER),
    "--output-dir", str(OUTPUT),
], check=True)


## 5. 读取结论与复现信息


In [ ]:
result = json.loads((OUTPUT / "result.json").read_text(encoding="utf-8"))
manifest = json.loads((OUTPUT / "run_manifest.json").read_text(encoding="utf-8"))
verdict = result["verdict"]
for key in ("functional_pass", "performance_valid", "performance_beneficial"):
    print(f"{key}: {verdict[key]}")
print("execution_order:", manifest["execution_order"])
print("comparison_scope:", manifest["comparison_scope"])
print("commit:", manifest["commit"])


`functional_pass` 说明两套实现功能一致且没有 fallback；`performance_valid` 说明三轮数据可以比较；`performance_beneficial` 才说明实现包 B 在本次 workload 下更快。三者不能合并成一个结论，Path B 变慢也不影响功能通过。

旧的 910B3/CANN 9.0、batch=1 decode 记录曾出现 Path B 约慢 4.28%，但它只是历史结果，不是当前提交证据。学生必须以本次 `run_manifest.json`、原始日志和 `result.json` 为准。

## 课后实践

解释为什么调用次数更少仍可能在小 batch 解码中不占优，并分别从 dtype/Cast、布局、固定 launch 开销和统计顺序提出可证伪的后续实验。


In [ ]:
# 完成练习后按需执行；Notebook 不会自动展开答案。
!cat answer/02.02_answer.md
